# Parte Específica — EDA do Pump it Up

Projeto Final AVD 2026.2 — Grupo 6. Responsável: **Gustavo**.

Dataset: [Pump it Up: Data Mining the Water Table (DrivenData)](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/). Classe: `status_group` (`functional`, `functional needs repair`, `non functional`).

> **Sobre o uso de IA neste notebook:** as células de código marcadas com `# [IA]` foram geradas com apoio de IA para *tornar a análise possível* (cálculos, tabelas e gráficos). Toda interpretação, discussão e decisão de tratamento é autoral do grupo e fica nas células "Interpretação autoral".

## 0. Pergunta de pesquisa e público-alvo

**Pergunta (versão inicial):** *Can you predict which water pumps are faulty?*

**Público-alvo:** gestor público (ex.: Ministério da Água da Tanzânia).

> **[Interpretação autoral — preencher]**

## Configuração

In [ ]:
# [IA] Claude Code – configuração (imports, caminhos, cores da classe e função para salvar figuras)
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sobe pastas até achar a raiz do repositório (onde está o requirements.txt),
# para o notebook funcionar não importa de onde o Jupyter foi aberto.
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "parte_especifica"
OUT_DIR = ROOT / "out" / "parte_especifica"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Ordem e cores fixas da classe, para todos os gráficos da EDA lerem igual.
# Paleta Okabe-Ito (acessível para daltonismo, Aula 05). Pode trocar à vontade.
STATUS_ORDEM = ["functional", "functional needs repair", "non functional"]
STATUS_CORES = {
    "functional": "#0072B2",
    "functional needs repair": "#E69F00",
    "non functional": "#D55E00",
}


def salvar(fig, nome):
    """Salva a figura em out/parte_especifica/<nome>.png."""
    fig.savefig(OUT_DIR / f"{nome}.png", dpi=150, bbox_inches="tight")


print("Raiz do projeto:", ROOT)

## Carga dos dados

In [ ]:
# [IA] Claude Code – leitura do dataset bruto (df_raw nunca é alterado)
df_raw = pd.read_csv(DATA_DIR / "raw" / "pump_it_up_train.csv")
df = df_raw.copy()
print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

## 1. Análise Estrutural

Tipos, dimensões, cardinalidade, duplicados, colunas constantes, colunas redundantes e consistência dos textos.

In [ ]:
# [IA] Claude Code – visão geral de cada coluna (tipo, nº de valores únicos, nulos e um exemplo)
resumo = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "n_unicos": df.nunique(),
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(2),
    "exemplo": df.iloc[0],
})
resumo.sort_values("n_unicos")

In [ ]:
# [IA] Claude Code – separação das colunas em grupos usados no resto do notebook
COL_ID = "id"
COL_ALVO = "status_group"

# Medidas quantitativas
COLS_NUM = ["amount_tsh", "gps_height", "longitude", "latitude",
            "num_private", "population", "construction_year"]
# Numéricas no arquivo, mas são códigos de localização (na prática, qualitativas)
COLS_CODIGO = ["region_code", "district_code"]
COLS_DATA = ["date_recorded"]
# Todo o resto (exceto id e alvo) é qualitativo
COLS_CAT = [c for c in df.columns
            if c not in COLS_NUM + COLS_CODIGO + COLS_DATA + [COL_ID, COL_ALVO]]

print(f"Quantitativas ({len(COLS_NUM)}): {COLS_NUM}")
print(f"Códigos numéricos ({len(COLS_CODIGO)}): {COLS_CODIGO}")
print(f"Data ({len(COLS_DATA)}): {COLS_DATA}")
print(f"Qualitativas ({len(COLS_CAT)}): {COLS_CAT}")

In [ ]:
# [IA] Claude Code – linhas duplicadas, ids repetidos e colunas com um único valor
print("Linhas duplicadas (todas as colunas):", df.duplicated().sum())
print("Linhas duplicadas (ignorando o id):", df.drop(columns=COL_ID).duplicated().sum())
print("Ids repetidos:", df[COL_ID].duplicated().sum())

constantes = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
print("Colunas com um único valor:", constantes)
for c in constantes:
    print(f"  {c}: {df[c].unique()}")

### 1.1 Colunas redundantes

Várias colunas aparecem em "famílias" (ex.: `extraction_type`, `extraction_type_group`, `extraction_type_class`). A tabela abaixo verifica, para cada par, se a coluna mais geral é só um agrupamento da mais detalhada: cada categoria da detalhada cai sempre em **uma única** categoria da geral.

In [ ]:
# [IA] Claude Code – verifica se as colunas "irmãs" são agrupamentos umas das outras
FAMILIAS = {
    "extraction_type": ["extraction_type", "extraction_type_group", "extraction_type_class"],
    "quantity": ["quantity", "quantity_group"],
    "payment": ["payment", "payment_type"],
    "water_quality": ["water_quality", "quality_group"],
    "source": ["source", "source_type", "source_class"],
    "waterpoint_type": ["waterpoint_type", "waterpoint_type_group"],
    "management": ["management", "management_group"],
}

linhas = []
for familia, cols in FAMILIAS.items():
    for detalhada, geral in zip(cols[:-1], cols[1:]):
        # maior nº de categorias "gerais" associadas a uma mesma categoria "detalhada"
        max_gerais = df.groupby(detalhada, dropna=False)[geral].nunique(dropna=False).max()
        linhas.append({
            "familia": familia,
            "detalhada": detalhada,
            "geral": geral,
            "n_cat_detalhada": df[detalhada].nunique(),
            "n_cat_geral": df[geral].nunique(),
            "geral_e_agrupamento_da_detalhada": max_gerais == 1,
        })
pd.DataFrame(linhas)

In [ ]:
# [IA] Claude Code – exemplos lado a lado: como as categorias se correspondem
display(pd.crosstab(df["quantity"], df["quantity_group"]))
display(pd.crosstab(df["payment"], df["payment_type"]))

### 1.2 Consistência entre `region` e `region_code`

In [ ]:
# [IA] Claude Code – quantos códigos existem por região e quantas regiões por código
codigos_por_regiao = df.groupby("region")["region_code"].unique().apply(sorted)
print("Regiões com mais de um region_code:")
display(codigos_por_regiao[codigos_por_regiao.apply(len) > 1])

regioes_por_codigo = df.groupby("region_code")["region"].unique().apply(sorted)
print("region_code com mais de uma região:")
display(regioes_por_codigo[regioes_por_codigo.apply(len) > 1])

### 1.3 Alta cardinalidade e grafias diferentes do mesmo nome

In [ ]:
# [IA] Claude Code – nº de categorias antes e depois de padronizar maiúsculas/espaços
linhas = []
for col in ["funder", "installer", "wpt_name", "subvillage", "scheme_name", "ward", "lga"]:
    s = df[col].dropna().astype(str)
    linhas.append({
        "coluna": col,
        "n_unicos": s.nunique(),
        "n_unicos_minusculo_sem_espacos": s.str.strip().str.lower().nunique(),
        "top_5": s.value_counts().head(5).to_dict(),
    })
pd.DataFrame(linhas)

> **[Interpretação autoral — preencher]**

## 2. Qualidade dos dados e mecanismo de ausência

Faltantes explícitos (`NaN`), zeros e códigos que podem esconder faltantes, e como a ausência se distribui (entre colunas, por região, por ano de registro e por classe). Essa é a base para discutir o mecanismo (MCAR, MAR ou MNAR, Aula 09) antes de decidir qualquer tratamento.

**Nada é alterado em `df` nesta seção.**

In [ ]:
# [IA] Claude Code – faltantes explícitos (NaN)
nulos = df.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
pd.DataFrame({"n_nulos": nulos, "pct": (nulos / len(df) * 100).round(2)})

In [ ]:
# [IA] Claude Code – zeros, valores negativos e códigos que podem esconder faltantes
suspeitos = {
    "population == 0": df["population"] == 0,
    "construction_year == 0": df["construction_year"] == 0,
    "gps_height == 0": df["gps_height"] == 0,
    "gps_height < 0": df["gps_height"] < 0,
    "longitude == 0": df["longitude"] == 0,
    # a Tanzânia inteira fica no hemisfério sul, então latitude ~0 é suspeita
    "latitude > -0.001": df["latitude"] > -0.001,
    "amount_tsh == 0": df["amount_tsh"] == 0,
    "num_private == 0": df["num_private"] == 0,
    "funder in {'0', '-'}": df["funder"].isin(["0", "-"]),
    "installer in {'0', '-'}": df["installer"].isin(["0", "-"]),
}
pd.DataFrame({
    "n": {k: int(v.sum()) for k, v in suspeitos.items()},
    "pct": {k: round(v.mean() * 100, 2) for k, v in suspeitos.items()},
})

In [ ]:
# [IA] Claude Code – verificação: longitude == 0 e latitude ~0 acontecem nas mesmas linhas?
pd.crosstab(df["longitude"] == 0, df["latitude"] > -0.001,
            rownames=["longitude == 0"], colnames=["latitude > -0.001"])

In [ ]:
# [IA] Claude Code – gráfico: % de ausência por coluna (NaN explícito e zero suspeito)
pct_nan = (df.isna().mean() * 100)[lambda s: s > 0]
pct_zero = pd.Series({c: (df[c] == 0).mean() * 100
                      for c in ["population", "construction_year", "gps_height", "longitude"]})
graf = pd.concat([
    pd.DataFrame({"coluna": pct_nan.index, "pct": pct_nan.values, "tipo": "NaN explícito"}),
    pd.DataFrame({"coluna": pct_zero.index, "pct": pct_zero.values, "tipo": "Zero suspeito"}),
]).sort_values("pct", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=graf, y="coluna", x="pct", hue="tipo", dodge=False, ax=ax)
ax.set_title("Percentual de valores ausentes por coluna")
ax.set_xlabel("% das linhas")
ax.set_ylabel("Coluna")
ax.legend(title="Tipo de ausência")
salvar(fig, "02_ausencia_por_coluna")
plt.show()

### 2.1 Indicadores de ausência

Uma coluna 0/1 por variável (1 = ausente, contando `NaN` e os códigos suspeitos acima). Usada só para estudar o mecanismo.

In [ ]:
# [IA] Claude Code – indicadores de ausência (1 = ausente)
ausente = pd.DataFrame({
    "population": df["population"] == 0,
    "construction_year": df["construction_year"] == 0,
    "gps_height": df["gps_height"] == 0,
    "longitude": df["longitude"] == 0,
    "scheme_name": df["scheme_name"].isna(),
    "scheme_management": df["scheme_management"].isna(),
    "installer": df["installer"].isna() | df["installer"].isin(["0", "-"]),
    "funder": df["funder"].isna() | df["funder"].isin(["0", "-"]),
    "public_meeting": df["public_meeting"].isna(),
    "permit": df["permit"].isna(),
}).astype(int)
ausente.mean().mul(100).round(2).sort_values(ascending=False)

In [ ]:
# [IA] Claude Code – gráfico: a ausência de uma coluna acompanha a de outra? (correlação entre os indicadores)
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(ausente.corr(), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            cbar_kws={"label": "Correlação entre indicadores de ausência"}, ax=ax)
ax.set_title("Coocorrência de ausência entre colunas")
ax.set_xlabel("Indicador de ausência")
ax.set_ylabel("Indicador de ausência")
salvar(fig, "02_coocorrencia_ausencia")
plt.show()

In [ ]:
# [IA] Claude Code – gráfico: % de ausência de cada coluna dentro de cada região
por_regiao = ausente.groupby(df["region"]).mean().mul(100)
por_regiao = por_regiao.sort_values("construction_year", ascending=False)

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(por_regiao, annot=True, fmt=".0f", cmap="Greys",
            cbar_kws={"label": "% de linhas ausentes na região"}, ax=ax)
ax.set_title("Ausência por região (%)")
ax.set_xlabel("Coluna")
ax.set_ylabel("Região")
salvar(fig, "02_ausencia_por_regiao")
plt.show()

In [ ]:
# [IA] Claude Code – % de ausência por ano de registro (date_recorded)
ano_registro = pd.to_datetime(df["date_recorded"]).dt.year
display(ano_registro.value_counts().sort_index().rename("n_linhas").to_frame().T)
ausente.groupby(ano_registro).mean().mul(100).round(1)

In [ ]:
# [IA] Claude Code – a ausência está associada à classe? (% de ausência dentro de cada status)
display(ausente.groupby(df[COL_ALVO]).mean().mul(100).round(1).reindex(STATUS_ORDEM))

# e o contrário: distribuição da classe entre linhas com e sem construction_year
pd.crosstab(ausente["construction_year"].map({0: "informado", 1: "ausente (0)"}),
            df[COL_ALVO], normalize="index").mul(100).round(1)[STATUS_ORDEM]

### Mecanismo de ausência e tratamento proposto

> **[Interpretação autoral — preencher]**

## 3. Análise Distribucional

Distribuição da classe, das variáveis quantitativas e das qualitativas.

Nos gráficos das quantitativas, os zeros suspeitos da seção 2 são **escondidos apenas para a visualização** (cópia `viz`), para não deformar o eixo. `df` continua intacto.

In [ ]:
# [IA] Claude Code – gráfico: distribuição da classe
contagem = df[COL_ALVO].value_counts().reindex(STATUS_ORDEM)

fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.bar(contagem.index, contagem.values, color=[STATUS_CORES[s] for s in contagem.index])
ax.bar_label(barras, labels=[f"{v:,} ({v / len(df):.1%})" for v in contagem.values])
ax.set_title("Distribuição da situação dos pontos d'água (status_group)")
ax.set_xlabel("Situação")
ax.set_ylabel("Número de pontos d'água")
salvar(fig, "03_distribuicao_classe")
plt.show()

In [ ]:
# [IA] Claude Code – cópia só para visualização: zeros suspeitos viram NaN
viz = df[COLS_NUM].copy()
for c in ["population", "construction_year", "gps_height"]:
    viz.loc[viz[c] == 0, c] = np.nan
viz.loc[df["longitude"] == 0, ["longitude", "latitude"]] = np.nan

estat = viz.describe().T
estat["assimetria"] = viz.skew()
estat["n_ausentes_na_viz"] = viz.isna().sum()
estat.round(2)

In [ ]:
# [IA] Claude Code – gráfico: histogramas das quantitativas
COLS_LOG = ["amount_tsh", "population", "num_private"]  # muito assimétricas: só valores > 0, eixo em log

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, COLS_NUM):
    dados = viz[col].dropna()
    usar_log = col in COLS_LOG
    if usar_log:
        dados = dados[dados > 0]
    sns.histplot(dados, bins=40, log_scale=usar_log, ax=ax, color="#4C72B0")
    ax.set_title(col + (" (> 0, eixo log)" if usar_log else ""))
    ax.set_xlabel(col)
    ax.set_ylabel("Frequência")
axes.flat[-1].axis("off")
fig.suptitle("Distribuição das variáveis quantitativas (zeros suspeitos ocultos)", fontsize=14)
fig.tight_layout()
salvar(fig, "03_histogramas_quantitativas")
plt.show()

In [ ]:
# [IA] Claude Code – gráfico: frequência das qualitativas de baixa cardinalidade (até 30 categorias)
cats_baixa = [c for c in COLS_CAT + COLS_CODIGO if df[c].nunique() <= 30]
print(len(cats_baixa), "colunas:", cats_baixa)

n_cols = 4
n_lin = int(np.ceil(len(cats_baixa) / n_cols))
fig, axes = plt.subplots(n_lin, n_cols, figsize=(20, 4.2 * n_lin))
for ax, col in zip(axes.flat, cats_baixa):
    valores = df[col].astype(object).fillna("(ausente)").astype(str)
    ordem = valores.value_counts().index
    sns.countplot(y=valores, order=ordem, ax=ax, color="#4C72B0")
    ax.set_title(col)
    ax.set_xlabel("Número de pontos d'água")
    ax.set_ylabel("")
for ax in axes.flat[len(cats_baixa):]:
    ax.axis("off")
fig.suptitle("Frequência das variáveis qualitativas (até 30 categorias)", fontsize=14)
fig.tight_layout()
salvar(fig, "03_frequencias_qualitativas")
plt.show()

In [ ]:
# [IA] Claude Code – gráfico: as 15 categorias mais frequentes das colunas de alta cardinalidade
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, col in zip(axes, ["funder", "installer", "lga"]):
    top = df[col].value_counts().head(15)
    sns.barplot(x=top.values, y=top.index, ax=ax, color="#4C72B0")
    ax.set_title(f"{col}: 15 mais frequentes (de {df[col].nunique():,})")
    ax.set_xlabel("Número de pontos d'água")
    ax.set_ylabel(col)
fig.tight_layout()
salvar(fig, "03_top_alta_cardinalidade")
plt.show()

In [ ]:
# [IA] Claude Code – gráfico: registros por ano (date_recorded)
ano_registro = pd.to_datetime(df["date_recorded"]).dt.year
contagem_ano = ano_registro.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(contagem_ano.index.astype(str), contagem_ano.values, color="#4C72B0")
ax.set_title("Número de registros por ano de registro (date_recorded)")
ax.set_xlabel("Ano de registro")
ax.set_ylabel("Número de registros")
salvar(fig, "03_registros_por_ano")
plt.show()

> **[Interpretação autoral — preencher]**

## 4. Análise Relacional (com `status_group`)

Relações entre as quantitativas, de cada variável com a classe e uma visão multivariada. Continua usando a cópia `viz` (zeros suspeitos ocultos) para as quantitativas.

In [ ]:
# [IA] Claude Code – gráfico: correlação de Spearman entre as quantitativas (pares com valores válidos)
corr = viz.corr(method="spearman")
mascara = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, mask=mascara, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            cbar_kws={"label": "Correlação de Spearman"}, ax=ax)
ax.set_title("Correlação de Spearman entre as variáveis quantitativas")
ax.set_xlabel("Variável")
ax.set_ylabel("Variável")
salvar(fig, "04_spearman_quantitativas")
plt.show()

In [ ]:
# [IA] Claude Code – variável derivada (Aula 26, feature creation): idade do ponto d'água no registro
viz["idade_no_registro"] = pd.to_datetime(df["date_recorded"]).dt.year - viz["construction_year"]
print("Idades negativas (construção depois do registro):", (viz["idade_no_registro"] < 0).sum())
viz["idade_no_registro"].describe().round(1)

In [ ]:
# [IA] Claude Code – gráfico: cada quantitativa por classe (boxplot)
cols_box = ["amount_tsh", "gps_height", "population", "construction_year", "idade_no_registro", "latitude"]
dados_box = viz[cols_box].assign(status=df[COL_ALVO])

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, col in zip(axes.flat, cols_box):
    d = dados_box[[col, "status"]].dropna()
    usar_log = col in COLS_LOG
    if usar_log:
        d = d[d[col] > 0]
    sns.boxplot(data=d, x="status", y=col, hue="status", order=STATUS_ORDEM,
                palette=STATUS_CORES, legend=False, ax=ax)
    if usar_log:
        ax.set_yscale("log")
    ax.set_title(col + (" (> 0, eixo log)" if usar_log else ""))
    ax.set_xlabel("Situação")
    ax.set_ylabel(col)
    ax.tick_params(axis="x", labelrotation=10)
fig.suptitle("Variáveis quantitativas por situação do ponto d'água", fontsize=14)
fig.tight_layout()
salvar(fig, "04_quantitativas_por_classe")
plt.show()

In [ ]:
# [IA] Claude Code – função: proporção de cada status dentro das categorias de uma coluna
def proporcao_status(coluna):
    """Tabela com o % de cada status por categoria (linhas somam 100) e o n de cada categoria."""
    valores = df[coluna].astype(object).fillna("(ausente)").astype(str)
    tabela = pd.crosstab(valores, df[COL_ALVO], normalize="index").mul(100)[STATUS_ORDEM]
    tabela["n"] = valores.value_counts()
    return tabela.sort_values("n", ascending=False)


proporcao_status("quantity").round(1)

In [ ]:
# [IA] Claude Code – gráfico: barras 100% empilhadas do status por categoria (versões agregadas das famílias)
cols_rel = ["quantity", "extraction_type_class", "waterpoint_type_group", "payment_type",
            "quality_group", "source_type", "management_group", "basin",
            "scheme_management", "public_meeting", "permit", "source_class"]

fig, axes = plt.subplots(4, 3, figsize=(20, 22))
for ax, col in zip(axes.flat, cols_rel):
    tabela = proporcao_status(col)
    tabela[STATUS_ORDEM].iloc[::-1].plot.barh(
        stacked=True, ax=ax, color=[STATUS_CORES[s] for s in STATUS_ORDEM], width=0.8, legend=False)
    ax.set_yticklabels([f"{i} (n={n:,})" for i, n in tabela["n"].iloc[::-1].items()])
    ax.set_title(col)
    ax.set_xlabel("% dos pontos d'água da categoria")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Situação", loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.01))
fig.suptitle("Situação do ponto d'água por categoria", fontsize=14, y=1.03)
fig.tight_layout()
salvar(fig, "04_status_por_categoria")
plt.show()

In [ ]:
# [IA] Claude Code – gráfico: % de cada status por região
tabela_regiao = proporcao_status("region")

fig, ax = plt.subplots(figsize=(9, 9))
sns.heatmap(tabela_regiao[STATUS_ORDEM], annot=True, fmt=".0f", cmap="Greys",
            cbar_kws={"label": "% dos pontos d'água da região"}, ax=ax)
ax.set_yticklabels([f"{r} (n={n:,})" for r, n in tabela_regiao["n"].items()], rotation=0)
ax.set_title("Situação dos pontos d'água por região (%)")
ax.set_xlabel("Situação")
ax.set_ylabel("Região")
salvar(fig, "04_status_por_regiao")
plt.show()

In [ ]:
# [IA] Claude Code – gráfico: localização dos pontos d'água por status (coordenadas válidas)
geo = viz[["longitude", "latitude"]].assign(status=df[COL_ALVO]).dropna()

fig, axes = plt.subplots(1, 3, figsize=(20, 7), sharex=True, sharey=True)
for ax, status in zip(axes, STATUS_ORDEM):
    ax.scatter(geo["longitude"], geo["latitude"], s=1, color="#DDDDDD", label="todos")
    sub = geo[geo["status"] == status]
    ax.scatter(sub["longitude"], sub["latitude"], s=2, alpha=0.4,
               color=STATUS_CORES[status], label=status)
    ax.set_title(f"{status} (n={len(sub):,})")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(loc="lower left", markerscale=6)
fig.suptitle("Localização dos pontos d'água por situação", fontsize=14)
fig.tight_layout()
salvar(fig, "04_mapa_por_status")
plt.show()

### 4.1 Visão multivariada

Função para cruzar **duas** variáveis qualitativas e ver o % de um status em cada combinação. Abaixo, um exemplo; troque as colunas e o status para explorar outras combinações.

In [ ]:
# [IA] Claude Code – função: % de um status em cada combinação de duas colunas qualitativas
def taxa_status(linha, coluna, status="non functional", n_min=50):
    """Heatmap do % de `status` em cada combinação (linha x coluna). Combinações com menos de n_min linhas ficam em branco."""
    grupos = df.groupby([linha, coluna])[COL_ALVO]
    taxa = grupos.apply(lambda s: (s == status).mean() * 100).unstack()
    n = grupos.size().unstack()
    taxa = taxa.where(n >= n_min)

    fig, ax = plt.subplots(figsize=(1.1 * taxa.shape[1] + 4, 0.5 * taxa.shape[0] + 2.5))
    sns.heatmap(taxa, annot=True, fmt=".0f", cmap="Greys", vmin=0, vmax=100,
                cbar_kws={"label": f"% {status}"}, ax=ax)
    ax.set_title(f"% '{status}' por {linha} x {coluna} (combinações com n >= {n_min})")
    ax.set_xlabel(coluna)
    ax.set_ylabel(linha)
    return fig, ax


fig, ax = taxa_status("quantity", "extraction_type_class")
salvar(fig, "04_multivariada_quantity_extraction")
plt.show()

> **[Interpretação autoral — preencher]**

## 5. Outliers

Identificação pelo critério do IQR (Aula 09), calculado sobre os valores válidos (cópia `viz`). Também listamos os maiores valores e casos impossíveis.

In [ ]:
# [IA] Claude Code – tabela de outliers pelo IQR (1,5 x IQR)
def resumo_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    fora = (s < lim_inf) | (s > lim_sup)
    return pd.Series({"n_validos": len(s), "q1": q1, "q3": q3, "iqr": iqr,
                      "lim_inf": lim_inf, "lim_sup": lim_sup, "min": s.min(), "max": s.max(),
                      "n_outliers": int(fora.sum()), "pct_outliers": round(fora.mean() * 100, 2)})


cols_out = ["amount_tsh", "gps_height", "population", "construction_year", "idade_no_registro", "num_private"]
pd.DataFrame({c: resumo_iqr(viz[c]) for c in cols_out}).T.round(2)

In [ ]:
# [IA] Claude Code – gráfico: boxplots das quantitativas (valores válidos)
fig, axes = plt.subplots(1, len(cols_out), figsize=(20, 5))
for ax, col in zip(axes, cols_out):
    d = viz[col].dropna()
    usar_log = col in COLS_LOG
    if usar_log:
        d = d[d > 0]
    sns.boxplot(y=d, ax=ax, color="#4C72B0")
    if usar_log:
        ax.set_yscale("log")
    ax.set_title(col + ("\n(> 0, eixo log)" if usar_log else ""))
    ax.set_xlabel("")
    ax.set_ylabel(col)
fig.suptitle("Boxplots das variáveis quantitativas (zeros suspeitos ocultos)", fontsize=14)
fig.tight_layout()
salvar(fig, "05_boxplots_outliers")
plt.show()

In [ ]:
# [IA] Claude Code – maiores valores de amount_tsh e population (para inspecionar os extremos)
colunas_ver = ["amount_tsh", "population", "region", "lga", "extraction_type_class",
               "waterpoint_type", "quantity", COL_ALVO]
display(df.nlargest(10, "amount_tsh")[colunas_ver])
display(df.nlargest(10, "population")[colunas_ver])

In [ ]:
# [IA] Claude Code – valores impossíveis ou estranhos
print("Construção depois do registro:", (viz["idade_no_registro"] < 0).sum())
print("gps_height negativo:", (df["gps_height"] < 0).sum(),
      "| mínimo:", df["gps_height"].min())
print("Regiões com gps_height negativo:")
df.loc[df["gps_height"] < 0, "region"].value_counts()

### Decisão sobre os outliers

> **[Interpretação autoral — preencher]**

## 6. Tratamento e exportação

Aplicar aqui **apenas** os tratamentos decididos e justificados nas seções 2 e 5, e exportar para `data/parte_especifica/clean/` (lido pelo dashboard).

Decisões pendentes (comentários da prof na planilha do checkpoint):
- [ ] Zeros de `population`, `construction_year`, `gps_height` e `longitude`/`latitude`: tratar como faltantes? Imputar como?
- [ ] `scheme_name` (48,5% ausente), `scheme_management`, `installer`, `funder`, `public_meeting`, `permit`: como tratar?
- [ ] `funder`/`installer` com `'0'` ou `'-'` e grafias diferentes do mesmo nome
- [ ] `num_private` (98,7% zero) e `recorded_by` (valor único): descartar?
- [ ] Colunas redundantes: qual versão de cada família manter?
- [ ] Alta cardinalidade (`wpt_name`, `subvillage`, `funder`, `installer`): manter, agrupar ou descartar?
- [ ] Outliers de `amount_tsh` e `population`
- [ ] Nomes de colunas sem acento/espaço (exigência do Streamlit)

In [ ]:
# TODO: tratamentos + df.to_csv(DATA_DIR / 'clean' / 'pump_it_up_clean.csv', index=False)

## 7. Síntese da EDA

> **[Interpretação autoral — preencher]**